In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

# Kaggle dataset path
BASE_PATH = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection"
TRAIN_DICOM_PATH = os.path.join(BASE_PATH, "train")
CSV_PATH = os.path.join(BASE_PATH, "train.csv")

# Working directory
WORK_PATH = "/kaggle/working/CliniScan"
IMAGE_SAVE_PATH = os.path.join(WORK_PATH, "images")
LABEL_SAVE_PATH = os.path.join(WORK_PATH, "labels")

os.makedirs(IMAGE_SAVE_PATH, exist_ok=True)
os.makedirs(LABEL_SAVE_PATH, exist_ok=True)

print("Working directory created.")

# Convert DICOM to JPG

In [ ]:
import pydicom
import cv2
import numpy as np
from tqdm import tqdm

image_ids = [f.replace(".dicom", "") for f in os.listdir(TRAIN_DICOM_PATH)]

for image_id in tqdm(image_ids):
    
    dicom_path = os.path.join(TRAIN_DICOM_PATH, image_id + ".dicom")
    save_path = os.path.join(IMAGE_SAVE_PATH, image_id + ".jpg")
    
    try:
        dicom = pydicom.dcmread(dicom_path)
        image = dicom.pixel_array.astype(np.float32)
        
        # Normalize to 0–255
        image = (image - image.min()) / (image.max() - image.min())
        image = (image * 255).astype(np.uint8)
        
        # Resize to 1024×1024
        image = cv2.resize(image, (1024, 1024))
        
        # Save as JPG
        cv2.imwrite(save_path, image, [cv2.IMWRITE_JPEG_QUALITY, 95])
        
    except Exception as e:
        print("Error:", image_id)

print("DICOM → JPG conversion completed.")

# Load CSV

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("CSV Loaded. Total Rows:", len(df))
df.head()

# Remove "No finding"

In [ ]:
df_filtered = df[df["class_name"] != "No finding"]

print("After removing 'No finding':", len(df_filtered))

In [ ]:
from tqdm import tqdm

# Group by image_id
grouped = df_filtered.groupby("image_id")

for image_id, group in tqdm(grouped):
    
    label_file_path = os.path.join(LABEL_SAVE_PATH, image_id + ".txt")
    
    # Image size after resizing
    img_w, img_h = 1024, 1024
    
    with open(label_file_path, "w") as f:
        
        for _, row in group.iterrows():
            
            class_id = int(row["class_id"])
            
            x_min = row["x_min"]
            y_min = row["y_min"]
            x_max = row["x_max"]
            y_max = row["y_max"]
            
            # Convert to YOLO format
            x_center = ((x_min + x_max) / 2) / img_w
            y_center = ((y_min + y_max) / 2) / img_h
            width = (x_max - x_min) / img_w
            height = (y_max - y_min) / img_h
            
            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

print("CSV → YOLO conversion completed.")

In [ ]:
all_images = set(df["image_id"].unique())
labeled_images = set(df_filtered["image_id"].unique())

normal_images = all_images - labeled_images

for image_id in normal_images:
    open(os.path.join(LABEL_SAVE_PATH, image_id + ".txt"), "w").close()

print("Empty label files created for normal images.")

In [ ]:
print("Total JPG Images:", len(os.listdir(IMAGE_SAVE_PATH)))
print("Total Label Files:", len(os.listdir(LABEL_SAVE_PATH)))

In [ ]:
!du -sh /kaggle/working/CliniScan

In [ ]:
!zip -r CliniScan_images.zip /kaggle/working/CliniScan/images
!zip -r CliniScan_labels.zip /kaggle/working/CliniScan/labels